## Dependency

In [ ]:
import sys
!{sys.executable} -m pip install transformers accelerate bitsandbytes einops
!{sys.executable} -m pip install langchain_huggingface langchain_community langgraph sentence_transformers chromadb pysqlite3-binary
!{sys.executable} -m pip install peft
!{sys.executable} -m pip install torch

In [ ]:
import sys
!{sys.executable} -m pip install datasets

## Constitutional Principles

In [ ]:
from pathlib import Path
import os

workspace = Path.cwd().resolve()
repo_root = next((p for p in (workspace, *workspace.parents) if (p / "configs/constitution.json").is_file()), None)
if repo_root is None:
    raise RuntimeError("Start the notebook kernel inside the cloned repository.")
os.chdir(repo_root / "CAI_Pipline")

import json

with (repo_root / "configs/constitution.json").open(encoding="utf-8") as f:
    constitution = json.load(f)

constitution["principles"][0]

In [ ]:
def get_applicable_principles(emotion: str, urgency: str, constitution: dict):
    """
    Select all applicable constitutional principles based on emotion + urgency.

    Args:
        emotion (str): ["angry", "frustrated", "anxious", "neutral", "satisfied", "happy"]
        urgency (str): ["low", "medium", "high"]
        constitution (dict): loaded constitution.json

    Returns:
        List of principle dicts that should be applied.
    """

    applicable = []

    #Extract mappings for convenience
    emo_map = constitution["emotion_category_mapping"]
    urg_map = constitution["urgency_mapping"]

    def emotion_matches(category: str):
        """Check if emotion belongs to a mapped category."""
        return emotion in emo_map.get(category, [])

    def urgency_matches(category: str):
        """Check if urgency belongs to a mapped urgency bucket."""
        return urgency in urg_map.get(category, [])

    for principle in constitution["principles"]:
        cond = principle["condition"]

        #Global match (always apply)
        if cond == "any":
            applicable.append(principle)
            continue

        #Emotion-based conditions
        if cond in emo_map and emotion_matches(cond):
            applicable.append(principle)
            continue

        #Urgency-based conditions
        if cond in urg_map and urgency_matches(cond):
            applicable.append(principle)
            continue

        #Special aliases in your constitution
        if cond == "emotion_negative_or_neutral" and emotion in emo_map["emotion_negative_or_neutral"]:
            applicable.append(principle)
            continue

        if cond == "emotion_positive" and emotion in emo_map["emotion_positive"]:
            applicable.append(principle)
            continue

        if cond == "emotion_high_tension" and emotion in emo_map["emotion_high_tension"]:
            applicable.append(principle)
            continue

    return applicable


In [ ]:
json_path = "./results_qwen.json"

with open(json_path, "r") as f:
    raw_examples = json.load(f)

## Base Model Set up


## Draft Reply from the Model

In [ ]:
#load model and tokenizer
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

#Base model name
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

#Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

#Load base model ONCE
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16,
)

#Path to your LoRA / fine-tuned adapters (folder you saved)
FINETUNED_PATH = "./qwen2-1.5b-finetuned"

#Load PEFT model
model = PeftModel.from_pretrained(
    base_model,
    FINETUNED_PATH,
)

In [ ]:
def generate_draft_reply(model, tokenizer, user_input: str, emotion_label: str, urgency_level: str, max_new_tokens: int = 300,):
    """
    Ask the model to generate draft output(pre-constitutional).

    Args:
        model: The Finetuned model
        tokenizer: Matching tokenizer.
        user_input: Raw user message.
        emotion_label: Detected emotion(angry, frustrated, anxious, neutral, satisfied, happy).
        urgency_level: Detected urgency(low, medium, high).
        max_new_tokens: Max tokens to generate
        temperature: Sampling temperature.
        top_p: Nucleus sampling parameter.

    Returns:
        draft_reply (str): The model's draft answer (no constitutional revision yet).
    """

    #Build up system prompt that ask the model to generate reponse according to the emotion label and urgent level
    system_prompt = (
        "You are a GENERAL helpful assistant, NOT a ticket agent"
        " NEVER mention company names, websites, or booking services"
        f"The user's detected emotion is '{emotion_label}'."
        f"The urgency level is {urgency_level} (1=low, 2=medium, 3=high)."
        "ANSWER THE QUESTION using ONLY the Context and prompt provided."
        "NEVER make up information. NEVER mention companies or websites."
        "Respond naturally and helpfully, but do NOT apply any extra constitutional self-critique or revision here. This is only the initial draft reply."
        "Maximum 2-3 sentences."
    )
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_input},
    ]

    #Set up chat template for the model
    prompt_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)

    #Track length to slice out only the generated part
    prompt_len = prompt_ids.shape[-1]

    #Build attention mask explicitly
    attention_mask = torch.ones_like(prompt_ids)

    output_ids = model.generate(
        prompt_ids,
        attention_mask=attention_mask, 
        max_new_tokens=max_new_tokens,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

    #Only decode the newly generated tokens (after the prompt)
    generated_ids = output_ids[0, prompt_len:]
    draft_reply = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    return draft_reply


In [ ]:
#define arguments
user_input=raw_examples[0]["text"]
emotion_label=raw_examples[0]["emotional"]
urgency_level=raw_examples[0]["urgency"]

print(f"user_input: {user_input}")
print(f"emotion_label: {emotion_label}")
print(f"urgency_level: {urgency_level}")

In [ ]:
get_applicable_principles(emotion_label, urgency_level, constitution)

In [ ]:
draft_reply = generate_draft_reply(model, tokenizer, user_input, emotion_label, urgency_level )

In [ ]:
print(draft_reply)

## Generate Critiques for Draft Reply

In [ ]:
def generate_critique(model, tokenizer, user_input: str, emotion: str, urgency: str, draft_reply: str, constitution: dict, max_new_tokens: int = 256,):
    """
    Generate critiques of a draft reply and the constitutional principles.

    Args:
        model: The fine-tuned model.
        tokenizer: Matching tokenizer.
        user_input: Raw user message.
        emotion_label: Detected emotion(angry, frustrated, anxious, neutral, satisfied, happy).
        urgency_level: Detected urgency(low, medium, high).
        draft_reply: The initial draft reply from the model.
        constitution: Loaded constitution.json (dict).
        max_new_tokens: Max tokens to generate

    Returns:
        List[dict]: each item like
            {
              "principle_name": ...,
              "condition": ...,
              "critic_instruction": ...,
              "critique": ...
            }
    """

    #Get the applicable principles corresponding to the emtion label and urgency level
    applicable_principles = get_applicable_principles(emotion, urgency, constitution)

    critiques = []

    for principle in applicable_principles:
        critic_instruction = principle["critic_instruction"].replace("<EMOTION>", emotion)
        #Build up system message
        system_prompt = (
            "You are an AI assistant that critiques another assistant's reply "
            "according to a specific principle. Be precise and concrete."
        )

        #Include all context needed for model to critique
        user_content = (
            f"User input:\n{user_input}\n\n"
            f"Detected emotion: {emotion}\n"
            f"Detected urgency: {urgency}\n\n"
            f"Assistant draft reply:\n{draft_reply}\n\n"
            f"Critique task (principle: {principle['name']}):\n"
            f"{critic_instruction}\n\n"
            "Now provide your critique:"
        )

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_content},
        ]

        #Set up chat template
        input_ids = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
        ).to(model.device)

        prompt_len = input_ids.shape[-1]

        output_ids = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,              # deterministic critiques
            pad_token_id=tokenizer.eos_token_id,
        )

        gen_ids = output_ids[0, prompt_len:]
        critique_text = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

        critiques.append({
            "principle_name": principle["name"],
            "condition": principle["condition"],
            "critic_instruction": critic_instruction,
            "critique": critique_text,
        })

    return critiques


In [ ]:
#test
critiques=generate_critique(model, tokenizer, user_input, emotion_label, urgency_level, draft_reply, constitution,)

In [ ]:
for c in critiques:
  print(c['principle_name'], c['critique'])

## Revise the Reply based on Critique

In [ ]:
def generate_revised_reply(model, tokenizer, user_input: str, emotion: str, urgency: str, draft_reply: str, critiques: list, constitution: dict, max_new_tokens: int = 300,):
    """
    Generate a revised reply using Qwen 2.5 based on critiques + constitutional principles.

    Args:
        model: The fine-tuned model.
        tokenizer: Matching tokenizer.
        user_input: Raw user message.
        emotion_label: Detected emotion(angry, frustrated, anxious, neutral, satisfied, happy).
        urgency_level: Detected urgency(low, medium, high).
        draft_reply: The initial draft reply from the model.
        critiques: List of critiques based on constition principles and user_input
        constitution: Loaded constitution.json (dict).
        max_new_tokens: Max tokens to generate

    Returns:
        revised_reply (str)
    """

    # Build a quick lookup: principle_name -> revision_instruction
    revision_instructions_map = {
        p["name"]: p["revision_instruction"]
        for p in constitution["principles"]
    }

    # Format critiques + revision instructions as guidance text
    critique_blocks = []
    for c in critiques:
        pname = c["principle_name"]
        critic_text = c["critique"]
        rev_inst = revision_instructions_map.get(pname, "")

        block = (
            f"Principle: {pname}\n"
            f"Critique: {critic_text}\n"
            f"Revision guidance: {rev_inst}\n"
        )
        critique_blocks.append(block)

    critiques_text = "\n\n".join(critique_blocks)

    system_prompt = (
        "You are an AI assistant that rewrites another assistant's reply "
        "according to multiple constitutional principles and critiques. "
        "Preserve the helpful content, but fix all issues pointed out in the critiques."
    )

    user_content = (
        f"User input:\n{user_input}\n\n"
        f"Detected emotion: {emotion}\n"
        f"Detected urgency: {urgency}\n\n"
        f"Original draft reply:\n{draft_reply}\n\n"
        f"Critiques and revision guidance:\n{critiques_text}\n\n"
        "Task: Rewrite the assistant's reply. "
        "The new reply should:\n"
        "- Address the user's input.\n"
        "- Respect the detected emotion and urgency.\n"
        "- Follow all revision guidance from the principles.\n"
        "Now provide ONLY the revised reply:"
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_content},
    ]

    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)

    prompt_len = input_ids.shape[-1]

    output_ids = model.generate(
        input_ids,
        max_new_tokens=max_new_tokens,
        do_sample=False,            # deterministic revision
        temperature=0.0,
        pad_token_id=tokenizer.eos_token_id,
    )

    gen_ids = output_ids[0, prompt_len:]
    revised_reply = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

    return revised_reply


In [ ]:
#test
generate_revised_reply(model, tokenizer, user_input, emotion_label, urgency_level, draft_reply, critiques, constitution)

In [ ]:
def generate_cai_pair(model, tokenizer, user_input: str, emotion_label: str, urgency_level: str, constitution: dict,):
    """
    Draft and Revise Pair Generation

    1) Generate draft reply (pre-constitutional).
    2) Generate critiques using constitutional principles.
    3) Generate revised reply using critiques + principles.

    Args:
        model: The finetuned model.
        tokenizer: Matching tokenizer.
        user_input: Raw user message.
        emotion_label: ["angry", "frustrated", "anxious", "neutral", "satisfied", "happy"].
        urgency_level: ["low", "medium", "high"].
        constitution: loaded constitution.json dict.

    Returns:
        A dict with:
            - user_input
            - emotion_label
            - urgency_level
            - draft_reply
            - revised_reply
            - critiques (list of per-principle critiques)
    """

    # 1) Draft reply
    draft_reply = generate_draft_reply(
        model=model,
        tokenizer=tokenizer,
        user_input=user_input,
        emotion_label=emotion_label,
        urgency_level=urgency_level,
    )

    # 2) Critiques based on constitution
    critiques = generate_critique(
        model=model,
        tokenizer=tokenizer,
        user_input=user_input,
        emotion=emotion_label,
        urgency=urgency_level,
        draft_reply=draft_reply,
        constitution=constitution,
    )

    # 3) Revised reply
    revised_reply = generate_revised_reply(
        model=model,
        tokenizer=tokenizer,
        user_input=user_input,
        emotion=emotion_label,
        urgency=urgency_level,
        draft_reply=draft_reply,
        critiques=critiques,
        constitution=constitution,
    )
    
    return {
        "user_input": user_input,
        "emotion_label": emotion_label,
        "urgency_level": urgency_level,
        "draft_reply": draft_reply,
        "revised_reply": revised_reply,
        "critiques": critiques,
    }


In [ ]:
#define arguments
user_input=raw_examples[0]["text"]
emotion_label=raw_examples[0]["emotional"]
urgency_level=raw_examples[0]["urgency"]
#test generate_cai_pair function
cair_pair = generate_cai_pair(model, tokenizer, user_input, emotion_label, urgency_level, constitution)

In [ ]:
print(cair_pair['user_input'])
print(cair_pair['draft_reply'])
print(cair_pair['revised_reply'])

## Create Draft-Revised Pair Dataset

In [ ]:
import json
from tqdm import tqdm

def build_cai_pairs_jsonl(model, tokenizer, raw_examples, constitution, out_path="cai_pairs.jsonl"):
    """
    raw_examples: [
      {"user_input": "...", "emotion": "frustrated", "urgency": "medium"},
      ...
    ]
    """
    with open(out_path, "w") as f:
        for ex in tqdm(raw_examples):

            # skip if user text is missing / null / empty
            if ex["text"] is None or str(ex["text"]).strip() == "":
                # optional: print or log to see how many you skip
                # print("Skipping example with null text:", ex.get("index"))
                continue
                
            
            pair = generate_cai_pair(
                model=model,
                tokenizer=tokenizer,
                #map JSON fields to arguments
                user_input=ex["text"],
                emotion_label=ex["emotional"],
                urgency_level=ex["urgency"],
                constitution=constitution,
            )
            # Save only what we need + keep extras if useful later
            json_line = {
                "user_input": pair["user_input"],
                "emotion": pair["emotion_label"],
                "urgency": pair["urgency_level"],
                "draft_reply": pair["draft_reply"],
                "revised_reply": pair["revised_reply"],
                "critiques": pair["critiques"],
            }
            f.write(json.dumps(json_line, ensure_ascii=False) + "\n")

    print("Saved CAI pairs to", out_path)


In [ ]:
build_cai_pairs_jsonl(model, tokenizer, raw_examples, constitution)

## Set Up Message for finetuning

In [ ]:
from datasets import load_dataset

ds = load_dataset("json", data_files="cai_pairs.jsonl")["train"]

def add_messages(example):
    #retrieve emtion and urgency from example
    emotion = example['emotion']
    urgency = example['urgency']
    
    system_msg = (
        "You are a GENERAL helpful assistant, NOT a ticket agent"
        " NEVER mention company names, websites, or booking services"
        f"The user's detected emotion is '{emotion}'."
        f"The urgency level is {urgency} (1=low, 2=medium, 3=high)."
        "ANSWER THE QUESTION using ONLY the Context and prompt provided."
        "NEVER make up information. NEVER mention companies or websites."
        "Respond naturally and helpfully, but do NOT apply any extra constitutional self-critique or revision here. This is only the initial draft reply."
        "Maximum 2-3 sentences."
    )

    example["messages"] = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": example["user_input"]},
        {"role": "assistant", "content": example["revised_reply"]},
    ]
    return example

ds = ds.map(add_messages)

orig_cols = ds.column_names
print(orig_cols)


## Tokenize the data

In [ ]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize_chat(example):
    # Use Qwen chat template (messages must already be there)
    text_ids = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=True,
        add_generation_prompt=False,
        return_tensors=None,      # list[int]
    )

    example["input_ids"] = text_ids
    example["attention_mask"] = [1] * len(text_ids)
    return example

tokenized_ds = ds.map(
    tokenize_chat,
    remove_columns=orig_cols,   # drop JSON fields and messages, keep only model fields we just added
)

print(tokenized_ds[0])

## Supervised Fine Tuning Step

In [ ]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer,
)
from peft import PeftModel
import torch

BASE_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_PATH = "./qwen2-1.5b-finetuned"    # LoRA adapter dir

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    device_map="auto",
    dtype=torch.float16,
)

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH,
)

# Make sure LoRA params are trainable
model.config.use_cache = False
model.train()
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

model.print_trainable_parameters()
print("Any trainable params?", any(p.requires_grad for p in model.parameters()))

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

training_args = TrainingArguments(
    output_dir="qwen2.5-cai-sft",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    learning_rate=2e-5,
    warmup_ratio=0.03,
    weight_decay=0.01,
    logging_steps=10,
    save_steps=500,
    save_total_limit=2,
    bf16=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds,
    data_collator=data_collator,
)

trainer.train()

# Save updated adapter (and optionally full merged model if you merge later)
model.save_pretrained("qwen2.5-cai-sft-final")
tokenizer.save_pretrained("qwen2.5-cai-sft-final")


## Judge Model Set Up

In [ ]:
import sys
!{sys.executable} -m pip install --upgrade openai

In [ ]:
import os
from openai import OpenAI


client = OpenAI()   # Client automatically reads the environment variable
print("Client initialized.")

## DPO Trianing Data Set Up

#### LLM-as-a-Judge Preference Labeling for DPO.

In [ ]:
import json
from datasets import Dataset
from tqdm import tqdm

def make_dpo_example_with_judge(cai_pair: dict,
                                judge_model: str = "gpt-4o-mini") -> dict:
    """
    Use a powerful reference model (gpt-4o-mini) to decide which reply
    is better between draft_reply and revised_reply, based on:
      - system prompt (emotion + urgency)
      - user input
    and return a single DPO example.
    """

    # 1. Extract fields from CAI pair
    user_input   = cai_pair["user_input"]
    emotion      = cai_pair["emotion"]
    urgency      = cai_pair["urgency"]
    draft_reply  = cai_pair["draft_reply"]
    revised_reply = cai_pair["revised_reply"]

    # 2. Build system prompt (same style as your SFT prompt)
    system_msg = (
        "You are a GENERAL helpful assistant, NOT a ticket agent.\n"
        "NEVER mention company names, websites, or booking services.\n"
        f"The user's detected emotion is '{emotion}'.\n"
        f"The urgency level is '{urgency}'.\n"
        "ANSWER THE QUESTION using ONLY the context and prompt provided.\n"
        "NEVER make up information. NEVER mention companies or websites.\n"
        "Respond naturally and helpfully.\n"
        "Maximum 2–3 sentences."
    )

    # 3. Build judge prompt
    judge_prompt = f"""
You are an expert evaluator for conversational AI systems.
Your job is to choose which assistant reply is better.

Evaluation criteria:
- Alignment with the system prompt
- Tone matches emotion: '{emotion}'
- Urgency handling matches level: '{urgency}'
- No hallucinated assumptions
- Concise, friendly, and helpful

SYSTEM PROMPT GIVEN TO THE MODEL:
\"\"\"{system_msg}\"\"\"

USER INPUT:
\"\"\"{user_input}\"\"\"

RESPONSE A (Draft):
\"\"\"{draft_reply}\"\"\"

RESPONSE B (Revised):
\"\"\"{revised_reply}\"\"\"

Which reply is better? Answer with ONLY a single character: 'A' or 'B'.
"""

    # 4. Ask judge model
    resp = client.chat.completions.create(
        model=judge_model,
        messages=[{"role": "user", "content": judge_prompt}],
        max_tokens=1,
        temperature=0,
    )

    decision = resp.choices[0].message.content.strip().upper()

    # Fallback safety
    if decision not in ("A", "B"):
        decision = "B"

    if decision == "A":
        chosen, rejected = draft_reply, revised_reply
    else:
        chosen, rejected = revised_reply, draft_reply

    # 5. Build DPO training prompt (same format as before)
    prompt = (
        f"System: {system_msg}\n"
        f"User: {user_input}\n"
        "Assistant:"
    )

    return {
        "prompt": prompt,
        "chosen_response": chosen,
        "rejected_response": rejected,
    }


In [ ]:
def load_cai_pairs_and_build_dpo_dataset_with_judge(
    cai_path: str,
    judge_model: str = "gpt-4o-mini",
) -> Dataset:
    """
    Read cai_pairs JSONL and use gpt-4o-mini to label each pair
    with chosen / rejected responses for DPO.
    """
    cai_pairs = []
    with open(cai_path, "r") as f:
        for line in f:
            if not line.strip():
                continue
            cai_pairs.append(json.loads(line))

    dpo_examples = []
    for p in tqdm(cai_pairs, desc="Labeling with judge model"):
        dpo_examples.append(make_dpo_example_with_judge(p, judge_model))

    dpo_ds = Dataset.from_list(dpo_examples)
    return dpo_ds


In [ ]:
dpo_ds = load_cai_pairs_and_build_dpo_dataset_with_judge("cai_pairs.jsonl")
print(dpo_ds[0])

## DPO Training Process

In [ ]:
base_model_name = "Qwen/Qwen2.5-1.5B-Instruct"
sft_lora_path  = "qwen2.5-cai-sft-final"

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Policy model = base model + your SFT LoRA
base_policy = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    device_map="auto",
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
)
policy_model = PeftModel.from_pretrained(
    base_policy,
    sft_lora_path,
)

# Reference model = frozen base model (no LoRA, no grad)
ref_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    device_map="auto",
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
)
ref_model.requires_grad_(False)   # important for DPO


In [ ]:
import sys
!{sys.executable} -m pip install --upgrade trl

In [ ]:
from trl import DPOTrainer, DPOConfig

# Make sure columns match what DPOTrainer expects
print(dpo_ds.column_names)  # optional sanity check

dpo_ds = dpo_ds.rename_columns({
    "chosen_response": "chosen",
    "rejected_response": "rejected",
})

training_args = DPOConfig(
    output_dir="qwen2.5-cai-dpo",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    learning_rate=2e-5,
    logging_steps=10,
    save_steps=200,
    report_to="none",
    beta=1.0,
    max_length=512,
    max_prompt_length=256,
)

# rename columns (as above) here

dpo_trainer = DPOTrainer(
    model=policy_model,
    ref_model=ref_model,
    args=training_args,
    train_dataset=dpo_ds,
    processing_class=tokenizer,
)

dpo_trainer.train()

dpo_output_dir = "qwen2.5-cai-dpo-final"
dpo_trainer.save_model(dpo_output_dir)
tokenizer.save_pretrained(dpo_output_dir)
print("DPO training finished. Saved to:", dpo_output_dir)
